In [ ]:
# Task 3: Build a Chatbot using Hugging Face Transformers

!pip install transformers torch

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# 2. Model Loading
# Using DialoGPT-medium for a balance of speed and conversational quality
model_name = "microsoft/DialoGPT-medium"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

def start_chatbot():
    print("Chatbot: Hello! I am your AI assistant. How can I help you today? (Type 'exit' or 'quit' to stop)")

    # Initialize chat history
    chat_history_ids = None

    while True:
        # 3. User Input Handling
        user_input = input("User: ")

        # 4. Exit Condition
        if user_input.lower() in ['exit', 'quit']:
            print("Chatbot: Goodbye! Have a great day.")
            break

        # 5. Response Generation (Processing)
        # Encode user input and add the end-of-sentence token
        new_user_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')

        # Create attention mask to improve response reliability
        attention_mask = torch.ones(new_user_input_ids.shape, dtype=torch.long)

        # Append new input to chat history for continuous conversation
        if chat_history_ids is not None:
            bot_input_ids = torch.cat([chat_history_ids, new_user_input_ids], dim=-1)
            attention_mask = torch.cat([torch.ones(chat_history_ids.shape, dtype=torch.long), attention_mask], dim=-1)
        else:
            bot_input_ids = new_user_input_ids

        # Generate response with tuned parameters for better logic
        chat_history_ids = model.generate(
            bot_input_ids,
            attention_mask=attention_mask,
            max_length=1000,
            pad_token_id=tokenizer.eos_token_id,
            no_repeat_ngram_size=3,
            do_sample=True,
            top_k=50,      # Focused selection
            top_p=0.9,     # Quality diversity
            temperature=0.7 # Balanced creativity
        )

        # 6. Display Output
        # Decode only the newly generated tokens
        response = tokenizer.decode(chat_history_ids[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)
        print(f"Chatbot: {response}")

# Run the chatbot
if __name__ == "__main__":
    start_chatbot()

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Chatbot: Hello! I am your AI assistant. How can I help you today? (Type 'exit' or 'quit' to stop)
User: "Hey"
Chatbot: Thanks for the reminder.
User: "What is AI"
Chatbot: AIuly, I'm sure.
User: "exit"
Chatbot: I think you mean departement.
